# Trading Intelligent — CH02 v4.0
## Laboratoire 1 — Voir une série, diagnostiquer la stationnarité et comprendre l'ACF

**Auteur académique : Tawfik Masrour**

**Position dans le cours :** Chapitre 2 — *Comprendre la dépendance temporelle : stationnarité, mémoire, ARIMA et saisonnalité*.

### Question directrice
Comment reconnaître ce qui relève d'une structure temporelle stable, d'une tendance, d'une marche aléatoire ou d'une transformation nécessaire — sans confondre impression visuelle et diagnostic statistique ?

> **Contrat pédagogique du laboratoire.** On ne demande jamais au code de nous dire ce que nous n'avons pas encore défini. Chaque expérience suit le fil : **question → intuition → calcul → graphique → diagnostic → interprétation financière**.

> Ce notebook reprend intégralement les expériences pédagogiques du laboratoire historique correspondant, mais les replace dans le cours **Trading Intelligent** et les enrichit par des explications, contrôles et ponts vers la finance.


### 0. Avant la première ligne de code : quatre objets à ne jamais confondre

Une **série temporelle observée** est une suite de valeurs ordonnées dans le temps. Derrière elle, nous imaginons un **processus stochastique** qui aurait pu produire d'autres trajectoires.

Dans ce laboratoire :
- une **tendance** déplace progressivement le niveau moyen ;
- une **saisonnalité** répète une structure à période identifiable ;
- une **dépendance temporelle** signifie que le passé contient de l'information sur le présent ;
- un **bruit blanc** sert de référence : moyenne stable, variance stable et absence d'autocorrélation linéaire aux retards non nuls.

En finance, le **niveau d'un prix** et son **rendement** ne sont pas la même variable. Un prix peut être non stationnaire alors que ses variations ou rendements sont beaucoup plus proches d'un processus stationnaire.


### Mini-calcul mental — avant tout graphique

Considérons deux suites :

\[
A=(10,11,12,13,14),\qquad
B=(10,12,9,11,8).
\]

La première montre une progression presque déterministe. La seconde fluctue autour d'un niveau. Avant de calculer ADF ou ACF, nous devons apprendre à poser la bonne question : **la loi de la série semble-t-elle rester la même au cours du temps ?**

Le graphique viendra ensuite confirmer ou contredire cette première lecture — jamais la remplacer.



##**1️⃣ Setup & Introduction**
Nous allons générer une série temporelle synthétique "Fil Rouge" pour comprendre mécaniquement ce qu'est la non-stationnarité.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import acf, pacf, adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Configuration esthétique
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliothèques chargées.")

##2️⃣ **Exemple 0 : La Cible (Stationnarité Parfaite)**
Avant de soigner les malades, regardons à quoi ressemble un patient en bonne santé.

### Pourquoi commencer par un « patient sain » ?

Avant d'étudier ce qui casse la stationnarité, il faut avoir une référence. Pour une stationnarité faible, trois propriétés doivent rester stables :

\[
\mathbb E[X_t]=\mu,\qquad
\operatorname{Var}(X_t)=\sigma^2,\qquad
\operatorname{Cov}(X_t,X_{t-k})=\gamma(k).
\]

La dernière égalité signifie que la covariance dépend du **retard \(k\)**, pas de la date \(t\). Deux observations séparées de 3 jours doivent avoir la même relation statistique, que nous soyons en janvier ou en juin — si le régime n'a pas changé.


In [ ]:
# Génération d'un Bruit Blanc Gaussien
np.random.seed(42)
n = 200
white_noise = np.random.normal(0, 1, n)

# Visualisation
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Série Temporelle
ax[0].plot(white_noise, color='#10b981', linewidth=1)
ax[0].set_title("La Cible : Bruit Blanc Gaussien (Stationnaire)")
ax[0].axhline(0, color='black', linestyle='--')
ax[0].set_xlabel("Temps")

# Autocorrélation (ACF)
plot_acf(white_noise, lags=40, ax=ax[1], title="ACF du Bruit Blanc (Aucune Mémoire)")
plt.show()

##**📝 Analyse de l'Expert**
Pourquoi c'est parfait ?

**Moyenne constante** : Elle oscille autour de 0.

**Variance constante** : Pas "d'effet mégaphone".

**ACF vide** : Aucune barre ne dépasse significativement la zone bleue. Le passé ne prédit pas le futur.

**Objectif** : Toutes nos transformations futures (Différenciation, Log) auront pour but de ramener nos séries complexes à cet état précis.

##**3️⃣ Cas 1 : La Tendance Linéaire**
Le cas le plus simple : la série monte ou descend tout droit.

Créons une série simple définie par $y_t = 10 + 0.5 t + \epsilon_t$. C'est l'exemple parfait d'une série non-stationnaire déterministe.

### Ce que la tendance détruit exactement

Une tendance déterministe simple peut s'écrire

\[
X_t=a+bt+
arepsilon_t.
\]

Alors

\[
\mathbb E[X_t]=a+bt,
\]

qui dépend de \(t\). La moyenne n'est donc plus constante : la stationnarité faible est violée **même si le bruit \(
arepsilon_t\) est parfaitement stationnaire**.

C'est important en finance : observer une hausse durable d'un niveau de prix ne signifie pas automatiquement qu'il existe une autocorrélation exploitable des rendements.


In [ ]:
# Génération de la série
np.random.seed(42)
n = 200
t = np.arange(n)
bruit = np.random.normal(0, 2, n)  # Bruit blanc gaussien

# Modèle : Tendance Linéaire + Bruit
y_trend = 10 + 0.5 * t + bruit

df = pd.DataFrame({'Temps': t, 'Valeur': y_trend})

# Visualisation
plt.figure(figsize=(12, 5))
plt.plot(df['Temps'], df['Valeur'], label='Série Observée ($y_t$)', color='#3b82f6')
plt.plot(df['Temps'], 10 + 0.5*t, label='Tendance Théorique ($10 + 0.5t$)', color='red', linestyle='--')
plt.title("Exemple: Série à Tendance Linéaire (Non-Stationnaire)", fontsize=14)
plt.legend()
plt.show()

##📝 **Commentaire Expert**
**À l'œil nu**, cette série n'est pas stationnaire.
**Moyenne :** Elle augmente constamment avec le temps ($\mathbb{E}[y_t]$ dépend de $t$).
**Variance :** Elle semble stable autour de la tendance (homoscédastique), mais la moyenne mobile change.

**Problème :** Si on calcule la moyenne globale de cette série ($\bar{y} \approx 60$), elle ne représente rien, car la série était à 10 au début et à 110 à la fin.

##**Diagnostic 1 : Autocovariance & Autocorrélation**

Calculons la "mémoire" de la série. Si la série est non-stationnaire, l'autocorrélation va diminuer très lentement.

In [ ]:
# Calcul de l'ACF (Autocorrélation)
acf_values = acf(df['Valeur'], nlags=40)

# Visualisation du Corrélogramme
plt.figure(figsize=(12, 5))
plot_acf(df['Valeur'], lags=40, ax=plt.gca(), title="Autocorrélogramme (ACF) - Série Brute")
plt.xlabel("Lag (Retard $k$)")
plt.ylabel("Autocorrélation $\\rho(k)$")
plt.show()

# Affichage des premières valeurs
print(f"Autocorrélation au Lag 1 : {acf_values[1]:.4f}")
print(f"Autocorrélation au Lag 10: {acf_values[10]:.4f}")
print(f"Autocorrélation au Lag 20: {acf_values[20]:.4f}")
print(f"Autocorrélation au Lag 30: {acf_values[30]:.4f}")

##📝 **Interprétation du Corrélogramme**
Regardez le graphique ci-dessus : Les barres bleues diminuent très lentement (linéairement).

Même à un décalage de $k=20$, la corrélation reste forte et positive.C'est la signature typique d'une série non-stationnaire (tendance). La valeur d'hier prédit très bien celle d'aujourd'hui simplement parce qu'elles sont toutes les deux "hautes" à cause de la tendance.

## **La Solution : La Différenciation ($\Delta$)**
Pour supprimer la tendance linéaire, nous appliquons l'opérateur différence : $\Delta y_t = y_t - y_{t-1}$.

### Pourquoi la différence retire une tendance linéaire

Si

\[
X_t=a+bt+
arepsilon_t,
\]

alors

\[
\Delta X_t=X_t-X_{t-1}
=b+(
arepsilon_t-
arepsilon_{t-1}).
\]

Le terme \(a+bt\) a disparu et il reste une moyenne constante \(b\). Cette petite dérivation explique **pourquoi** la différence première peut stationnariser une tendance linéaire.

Mais elle ne doit jamais être appliquée mécaniquement : différencier une série déjà stationnaire peut créer artificiellement une structure MA et augmenter le bruit.


In [ ]:
df['Diff_1'] = df['Valeur'].diff()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(df['Valeur'], color='#3b82f6')
axes[0].set_title("Avant : Série Non-Stationnaire")

axes[1].plot(df['Diff_1'], color='#10b981')
axes[1].set_title(r"Après : Série Différenciée $\left(\Delta y_t\right)$") # Corrected line with LaTeX parentheses
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

##📝 **Analyse Mathématique**
Si $y_t = \alpha + \beta t + \epsilon_t$,

alors :
$$\Delta y_t = (\alpha + \beta t + \epsilon_t) - (\alpha + \beta(t-1) + \epsilon_{t-1})$$$$\Delta y_t = \beta + (\epsilon_t - \epsilon_{t-1})$$

**Résultat :** La tendance $t$ a disparu ! Il ne reste qu'une constante $\beta$ (la pente) et du bruit. La moyenne est devenue constante. Nous avons atteint la stationnarité.

##**Vérification Finale : ACF sur Série Différenciée**
Prouvons que la mémoire infinie a disparu.

In [ ]:
# On retire la première valeur (NaN après diff)
diff_series = df['Diff_1'].dropna()

# Plot ACF
plt.figure(figsize=(12, 5))
plot_acf(diff_series, lags=40, ax=plt.gca(), title="ACF - Série Différenciée (Stationnaire)")
plt.show()

##📝 **Conclusion**
Le changement est radical : L'autocorrélation chute immédiatement à 0 après le lag 1.

***Toutes les barres sont dans la zone bleue (intervalle de confiance = bruit blanc).***

**Mission accomplie :** La série est prête pour la modélisation ARIMA (plus tard).

##4️⃣ **Cas 2 : La Marche Aléatoire (Stochastique)**

**Pourquoi cet exemple ?**

Contrairement à la tendance linéaire (qui est une ligne droite prévisible), la Marche Aléatoire est imprévisible.

**Linéaire**: Je monte par exemple de 1 marche chaque jour.

**Marche Aléatoire** : "Je lance une pièce. Pile je monte, Face je descends."

C'est le modèle de base des cours de bourse. ***La tendance n'est pas dans le temps, elle est dans l'accumulation des hasards***.

### Marche aléatoire : même équation visuelle, autre nature

Une marche aléatoire

\[
X_t=X_{t-1}+
arepsilon_t
\]

donne, par itération,

\[
X_t=X_0+\sum_{j=1}^{t}
arepsilon_j.
\]

Si les innovations ont variance \(\sigma_
arepsilon^2\),

\[
\operatorname{Var}(X_t)=t\sigma_
arepsilon^2.
\]

La variance augmente donc avec le temps. Ici la non-stationnarité n'est pas une simple courbe déterministe à retirer : **les chocs s'accumulent définitivement**.

En finance, cette différence est essentielle : une tendance déterministe et une tendance stochastique n'impliquent ni les mêmes tests ni les mêmes prévisions.


##**Étape A : Génération et Visualisation Temporelle**
Nous simulons un processus où la valeur d'aujourd'hui est égale à la valeur d'hier plus un choc aléatoire ($y_t = y_{t-1} + \epsilon_t$).

In [ ]:
# Génération d'une Marche Aléatoire
np.random.seed(100)
# Cumsum transforme le bruit blanc en marche aléatoire
random_walk = np.cumsum(np.random.normal(0, 1, n))

plt.figure(figsize=(12, 5))

plt.plot(random_walk, color='#8b5cf6', label=r'Marche Aléatoire ($y_t = y_{t-1} + \epsilon_t$)')

plt.title("Exemple 2 : Marche Aléatoire (Variance Explosive)")
plt.legend()
plt.show()

##**Étape B : Diagnostic Visuel (La Signature ACF)**
Avant de lancer un test statistique, regardons la mémoire de la série. Pour une marche aléatoire, la mémoire est "parfaite" à court terme et ne s'estompe que très lentement.

In [ ]:
# Calcul et Affichage de l'ACF
plt.figure(figsize=(12, 5))
plot_acf(random_walk, lags=40, ax=plt.gca(), title="ACF - Marche Aléatoire (Signature typique)")
plt.xlabel("Lag (Retard $k$)")
plt.ylabel("Autocorrélation $\\rho(k)$")
plt.show()
# Affichage des premières valeurs
print(f"Autocorrélation au Lag 1 : {acf_values[1]:.4f}")
print(f"Autocorrélation au Lag 10: {acf_values[10]:.4f}")
print(f"Autocorrélation au Lag 20: {acf_values[20]:.4f}")
print(f"Autocorrélation au Lag 30: {acf_values[30]:.4f}")

##📝 **Interprétation Expert**
Observez bien le graphique ci-dessus (ACF) :Les barres partent de 1.0 et descendent très lentement.
Contrairement au bruit blanc (qui chute à 0 immédiatement), ici $\rho(1) \approx 0.98$, $\rho(2) \approx 0.97$...

C'est la signature visuelle d'une **Racine Unitaire**. La série a une "**mémoire infinie**" des chocs passés.

##**Étape C : Confirmation Statistique (Test ADF)**
L'intuition visuelle est forte, mais le test de Dickey-Fuller va trancher mathématiquement.

### Lire ADF sans réciter « p-value < 0,05 »

Le test Dickey–Fuller part de

\[
X_t=
ho X_{t-1}+
arepsilon_t.
\]

En soustrayant \(X_{t-1}\) :

\[
\Delta X_t=(
ho-1)X_{t-1}+
arepsilon_t
=\gamma X_{t-1}+
arepsilon_t.
\]

La racine unitaire \(
ho=1\) équivaut donc à \(\gamma=0\). L'hypothèse nulle du test est précisément la présence d'une racine unitaire.

Une petite p-value indique que les données sont difficiles à concilier avec cette hypothèse nulle. **Elle ne mesure pas la probabilité que la série soit stationnaire.**


In [ ]:
# Test de Dickey-Fuller (ADF)
result = adfuller(random_walk)

print("--- 📊 Résultat du Test ADF ---")
print(f"Statistique ADF : {result[0]:.4f}")
print(f"p-value         : {result[1]:.4f}")

print("\n--- 🧠 Verdict ---")
if result[1] > 0.05:
    print("❌ H0 non rejetée : La p-value est élevée (> 0.05).")
    print("👉 La série est NON-STATIONNAIRE (Présence confirmée d'une Racine Unitaire).")
else:
    print("✅ H0 rejetée : La série est Stationnaire.")

##📝 **Le Piège de la Marche Aléatoire**
Contrairement à la tendance linéaire, cette série n'a pas de direction prévisible. Elle "erre" sans but.

Sa variance augmente avec le temps ($Var = t \sigma^2$).

Le test ADF confirme formellement la non-stationnarité (p-value élevée).



##**Étape D : Le Remède (Différenciation)**
Comme pour la tendance, la solution magique est l'opérateur $\Delta$.

Si $$y_t = y_{t-1} + \epsilon_t$$

alors $$\Delta y_t = y_t - y_{t-1} = \epsilon_t$$ (Bruit Blanc).

In [ ]:
# Différenciation
diff_rw = pd.Series(random_walk).diff().dropna()

# Visualisation Preuve
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot Série Différenciée
axes[0].plot(diff_rw, color='#10b981')

axes[0].set_title(r"Série Différenciée ($\Delta y_t$)")

axes[0].axhline(0, color='black', linestyle='--', alpha=0.5)

# Plot ACF Différenciée
plot_acf(diff_rw, lags=40, ax=axes[1], title="ACF - Après Différenciation")

plt.show()

# Vérification ADF rapide
p_val_diff = adfuller(diff_rw)[1]
print(f"P-value après différenciation : {p_val_diff:.4e} (Stationnaire ✅)")

#**5️⃣ Cas 3 : La Tendance Quadratique (Nécessite $d=2$)**

Que se passe-t-il si la croissance accélère (ex: chute d'un corps **$y = \frac{1}{2}gt^2$**) ?


***Une seule différenciation ne suffira pas !***

In [ ]:
# Modèle : y_t = 0.05 * t^2 + bruit
quad_trend = 0.05 * (t**2) + np.random.normal(0, 5, n)

# Différenciation d'ordre 1
diff_1_quad = np.diff(quad_trend, n=1)

# Différenciation d'ordre 2 (La différence de la différence)
diff_2_quad = np.diff(diff_1_quad, n=1)

# Visualisation
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

ax[0].plot(quad_trend, color='#ef4444')
ax[0].set_title(r"Série Originale ($t^2$)")

ax[1].plot(diff_1_quad, color='#f97316')
ax[1].set_title(r"Diff $d=1$ : Reste une tendance linéaire !")

ax[2].plot(diff_2_quad, color='#10b981')
ax[2].set_title(r"Diff $d=2$ : Stationnaire !")
ax[2].axhline(0.1, color='red', linestyle='--', label='Accélération constante')

plt.show()

##**📝 Mathématiques en action**
$$y_t \approx t^2$$ **(Parabole)** $$\Delta y_t \approx 2t$$

**(Droite) $\rightarrow$ Encore non-stationnaire !**

 $$\Delta^2 y_t \approx 2$$

 **(Constante) $\rightarrow$ Stationnaire.**

 **Règle** : Si vous différenciez une fois et qu'il reste une tendance, différenciez encore ($d=2$). (Rarement plus de 2 en pratique).

#**6️⃣ Cas 4 : Tendance Exponentielle (Nécessite Log + $d=1$)**

C'est le cas le plus courant en économie (PIB, Population...).

La croissance est en pourcentage!

Ici, la variance explose avec le niveau (Effet Mégaphone).

***La différenciation seule ne suffit pas car elle ne corrige pas la variance.***

In [ ]:
# Modèle : y_t = exp(t/50)
exp_trend = np.exp(t / 40)

# Tentative 1 : Différenciation directe
diff_direct = np.diff(exp_trend, n=1)

# Tentative 2 : Log puis Différenciation
log_series = np.log(exp_trend)
diff_log = np.diff(log_series, n=1)

# Visualisation
fig, ax = plt.subplots(2, 2, figsize=(16, 10))

# Originale
ax[0,0].plot(exp_trend, color='#881337')
ax[0,0].set_title("Originale : Exponentielle")

# Diff Directe (ECHEC)
ax[0,1].plot(diff_direct, color='#ef4444')
ax[0,1].set_title(r"ÉCHEC : Différenciation seule (Variance explose)")

# Transformation Log
ax[1,0].plot(log_series, color='#3b82f6')
ax[1,0].set_title(r"Étape 1 : Logarithme (Linéarisation)")

# Diff du Log (SUCCES)
ax[1,1].plot(diff_log, color='#10b981')
ax[1,1].set_title(r"Étape 2 : Diff du Log (Stationnaire)")

plt.tight_layout()
plt.show()

# **7️⃣ Cas 5 : Au-delà du Logarithme (La Famille Box-Cox)**

En Data Science, face à une variance instable (hétéroscédasticité), on a souvent le **réflexe pavlovien** : *"Variance qui bouge ? $\rightarrow$ Je prends le Logarithme !"*

Cependant, le Logarithme est une transformation violente. Il peut être **trop fort** (écraser la dynamique) ou **inadapté** si la variance ne croît pas de manière exponentielle.

Il est crucial de maîtriser la famille des **Transformations de Puissance**, généralisée par Box & Cox. Voici les alternatives stratégiques :

## **1. La Transformation Racine Carrée ($\sqrt{y_t}$)**
* **Mathématique :** Stabilise la variance lorsque $\sigma^2$ est proportionnelle à la moyenne $\mu$.
* **Contexte Type :** Les données de **Comptage** (Loi de Poisson).
* **Exemples :**
    * Nombre de clients entrant dans un magasin par heure.
    * Nombre de défauts sur une chaîne de production.
    * Nombre d'appels reçus dans un call center.
    * *Pourquoi pas le Log ?* Le Log est trop agressif pour ces variations linéaires.

## **2. La Transformation Inverse ($1/y_t$)**
* **Mathématique :** Inverse totalement l'échelle.
* **Contexte Type :** Explosion très violente vers l'infini ou comportement hyperbolique.
* **Exemples :**
    * Durée de vie d'un composant radioactif.
    * Prix d'une commodité en pénurie extrême (flambée des prix).

## **3. La Transformation Logit ($\ln(\frac{y}{1-y})$)**
* **Mathématique :** Étire l'intervalle borné $[0, 1]$ vers l'infini $[-\infty, +\infty]$.
* **Contexte Type :** Données bornées (Pourcentages, Taux, Probabilités).
* **Exemples :**
    * Taux de chômage (borné entre 0% et 100%).
    * Taux de remplissage d'un hôtel.
    * Pourcentage de batterie.
    * *Le problème du Log :* Il ne gère pas la borne supérieure (100%). Le Logit permet d'utiliser ARIMA sur des données bornées sans "dépasser" les bornes lors de la prédiction.

---

## **4. L'Arme Automatique : La Transformation de Box-Cox**
Au lieu de deviner s'il faut prendre le Log, la Racine ou l'Inverse, nous laissons l'ordinateur optimiser un paramètre **$\lambda$** (lambda) pour maximiser la normalité des données.

La formule magique est :
$$y_t^{(\lambda)} = \begin{cases} \frac{y_t^\lambda - 1}{\lambda} & \text{si } \lambda \neq 0 \\ \ln(y_t) & \text{si } \lambda = 0 \end{cases}$$

> **Guide de lecture du $\lambda$ optimal :**
> * $\lambda \approx 1$ : **Pas de transformation** (Données déjà saines).
> * $\lambda \approx 0.5$ : **Racine Carrée** (Comptage / Poisson).
> * $\lambda \approx 0$ : **Logarithme** (Croissance Exponentielle).
> * $\lambda \approx -1$ : **Inverse** (Comportement hyperbolique).

In [ ]:
from scipy.stats import boxcox

# --- GÉNÉRATION D'UNE SÉRIE DE COMPTAGE (POISSON) ---
# Ici, la variance augmente linéairement avec la moyenne (sigma^2 = mu)
# C'est typique d'un nombre de clients qui augmente progressivement
np.random.seed(42)
t = np.linspace(10, 100, 200)
count_data = np.random.poisson(lam=t)

# --- TRANSFORMATIONS ---
sqrt_data = np.sqrt(count_data)      # Lambda = 0.5
log_data = np.log(count_data)        # Lambda = 0
inv_data = 1 / count_data            # Lambda = -1

# Box-Cox Automatique (Python trouve le meilleur Lambda tout seul)
boxcox_data, best_lambda = boxcox(count_data)

# --- VISUALISATION ---
fig, ax = plt.subplots(2, 2, figsize=(16, 10))

# 1. Série Originale (Variance croissante modérée)
ax[0,0].plot(count_data, color='#1e293b')
ax[0,0].set_title(r"Originale : Données de Comptage (Poisson)")

# 2. Racine Carrée (La solution théorique idéale pour Poisson)
ax[0,1].plot(sqrt_data, color='#10b981')
ax[0,1].set_title(r"Solution 1 : Racine Carrée ($\sqrt{y_t}, \lambda=0.5$)")
ax[0,1].text(0, sqrt_data.mean(), "Variance stabilisée ✅", color='green', fontweight='bold')

# 3. Logarithme (Trop fort ici)
ax[1,0].plot(log_data, color='#f59e0b')
ax[1,0].set_title(r"Solution 2 : Logarithme ($\ln(y_t), \lambda=0$)")
ax[1,0].text(0, log_data.mean(), "Écrase trop la dynamique ⚠️", color='orange')

# 4. Box-Cox Optimisé
ax[1,1].plot(boxcox_data, color='#8b5cf6')
ax[1,1].set_title(f"Solution 3 : Box-Cox Automatique ($\\lambda={best_lambda:.2f}$)")

plt.tight_layout()
plt.show()

print(f"--- VERDICT BOX-COX ---")
print(f"Le Lambda optimal trouvé est : {best_lambda:.4f}")
if 0.4 < best_lambda < 0.6:
    print("✅ Conclusion : Le modèle confirme qu'une RACINE CARRÉE est la meilleure transformation.")
elif -0.1 < best_lambda < 0.1:
    print("✅ Conclusion : Le modèle suggère un LOGARITHME.")
else:
    print("✅ Conclusion : Transformation intermédiaire spécifique.")

##**Etape suivante:**
 **Combiner Box-Cox + Différenciation**

 Maintenant que nous avons notre $\lambda$ optimal ($0.75$), nous allons préparer la série pour le modèle.!


C'est le moment crucial où l'on passe du diagnostic ("Le médecin a dit qu'il fallait une transformation de puissance 0.75") au traitement ("On prend le médicament").

Si $\lambda \approx 0.75$, cela signifie que la variance grandit, mais moins vite que linéairement.

Ce n'est ni une racine carrée, ni une série normale.

C'est entre les deux

.Ce qu'on fait ensuite ***(La "Pipeline" de Stationnarisation)***

La transformation Box-Cox ne règle que le problème de la Variance (l'effet mégaphone).

Elle ne règle pas le problème de la Tendance (la moyenne qui monte).

Il faut donc appliquer la séquence suivante :

Appliquer Box-Cox $\rightarrow$

On obtient une série à variance constante, mais qui monte toujours.

Appliquer la Différenciation ($d=1$) $\rightarrow$

On stabilise la moyenne.

Vérifier $\rightarrow$ Si c'est un bruit blanc, on peut modéliser.


In [ ]:
# --- ÉTAPE 1 : Application de la Transformation (Stabilisation Variance) ---
# La fonction boxcox nous a déjà renvoyé les données transformées dans 'boxcox_data'
# Rappel : y_trans = (y^lambda - 1) / lambda
y_variance_stabilisee = boxcox_data

# --- ÉTAPE 2 : Application de la Différenciation (Stabilisation Moyenne) ---
# Box-Cox a fixé l'écart-type, mais la courbe monte toujours (Tendance).
# On applique donc d=1 sur la série transformée.
y_finale_stationnaire = np.diff(y_variance_stabilisee, n=1)

# --- VISUALISATION DE LA "GUÉRISON" ---
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# 1. Le Malade (Originale)
ax[0].plot(count_data, color='#ef4444')
ax[0].set_title("1. Série Originale\n(Variance instable + Tendance)")

# 2. Le Premier Soin (Box-Cox)
ax[1].plot(y_variance_stabilisee, color='#f59e0b')
ax[1].set_title(f"2. Après Box-Cox ($\lambda={best_lambda:.2f}$)\n(Variance Stable, mais Tendance reste)")

# 3. La Guérison Complète (Box-Cox + Diff)
ax[2].plot(y_finale_stationnaire, color='#10b981')
ax[2].set_title("3. Après Box-Cox + Différenciation\n(Stationnaire !)")
ax[2].axhline(0, color='black', linestyle='--')

plt.show()

# --- VERIFICATION ACF ---
plt.figure(figsize=(10, 4))
plot_acf(y_finale_stationnaire, lags=40, ax=plt.gca(), title="Preuve Finale : ACF de la série transformée et différenciée")
plt.show()

##**La Recette Finale pour ARIMA** :

Si vous obtenez un $\lambda$ spécifique (ex: 0.75), votre modèle prédictif sera un ARIMA sur la série transformée.Transformez : $Z_t = BoxCox(Y_t, \lambda=0.75)$Différenciez : $W_t = Z_t - Z_{t-1}$Modélisez $W_t$ avec ARIMA.Note Importante pour la Prédiction :Une fois que le modèle aura prédit le futur, il faudra faire le chemin inverse (Inverse Box-Cox) pour revenir à l'échelle réelle (le nombre de clients) :$$Y_{pred} = (\lambda \cdot Z_{pred} + 1)^{1/\lambda}$$

##**📝 Conclusion du Lab**:

**Linéaire:** Ne touchez à rien ($d=0$).

**Tendance Linéaire / Marche Aléatoire** : Différenciez une fois ($d=1$).

A**ccélération (Quadratique)** : Différenciez deux fois ($d=2$).

**Croissance Exponentielle / Variance Instable** : Prenez le Logarithme, puis différenciez.

## 🔬 Enrichissement Trading Intelligent — prix versus rendements

Nous allons maintenant construire un prix synthétique par accumulation de rendements. L'objectif n'est pas de fabriquer un marché réaliste, mais de voir une idée fondamentale :

- le **prix** peut ressembler à une marche aléatoire ;
- les **rendements** peuvent rester beaucoup plus stables ;
- modéliser directement le niveau du prix avec un modèle supposant la stationnarité peut être incohérent.

Nous définissons d'abord le rendement simple :

\[
r_t=
rac{P_t-P_{t-1}}{P_{t-1}}.
\]

Et le rendement logarithmique :

\[
\ell_t=\log P_t-\log P_{t-1}.
\]

Pour de petites variations, les deux sont proches.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller

rng = np.random.default_rng(20261008)
n = 800
r = rng.normal(loc=0.0002, scale=0.01, size=n)
price = 100 * np.exp(np.cumsum(r))
logret = np.diff(np.log(price))

fig, axes = plt.subplots(2, 1, figsize=(11, 7))
axes[0].plot(price)
axes[0].set_title("Niveau de prix synthétique : accumulation des chocs")
axes[0].set_ylabel("Prix")
axes[0].grid(alpha=.2)

axes[1].plot(logret)
axes[1].axhline(0, color="black", lw=.8)
axes[1].set_title("Rendements logarithmiques : fluctuations autour d'un niveau stable")
axes[1].set_ylabel("Rendement")
axes[1].grid(alpha=.2)
plt.tight_layout()
plt.show()

for name, x in [("prix", price), ("rendements log", logret)]:
    stat, pvalue, *_ = adfuller(x, autolag="AIC")
    print(f"{name:16s} | ADF={stat:8.3f} | p-value={pvalue:.5f}")


### Lecture attendue

Le graphique ne suffit pas ; le test ne suffit pas non plus. Ils racontent ensemble une histoire cohérente : l'accumulation des chocs construit un niveau très persistant, alors que les incréments qui ont généré ce niveau restent centrés autour d'une moyenne stable.

> **Réflexe Trading Intelligent :** toujours demander *quelle variable est réellement modélisée ?* Niveau, différence, rendement simple, log-rendement, volatilité ou spread ne portent pas la même information.


## ✅ Checklist de compréhension — Lab 1

À la fin de ce notebook, vous devez pouvoir expliquer sans Python :

1. ce que signifie stationnarité faible ;
2. pourquoi une tendance linéaire viole la constance de la moyenne ;
3. pourquoi une marche aléatoire viole aussi la constance de la variance ;
4. ce que fait mathématiquement une différence première ;
5. pourquoi ACF empirique et ACF théorique ne sont pas identiques ;
6. ce que teste réellement ADF ;
7. pourquoi un prix et un rendement ne doivent pas être traités comme la même série.

Si une de ces réponses reste floue, revenez à l'expérience correspondante avant de passer au Lab 2.
